# Circuit Debugger: train a terminal detector
Use a Colab GPU runtime. This notebook requires your **labeled** dataset; it does not turn arbitrary online pictures into training labels. Read `docs/DATASET.md` first. No trained results are included.

Classes: 0 resistor, 1 LED, 2 wire. Two keypoints per object: insertion points. For LEDs, point 1 is anode and point 2 cathode. Split by physical build/source to prevent leakage. Keep downloaded weights only from sources you trust.

This model supplies visual proposals; mapping them to breadboard holes and verifying connectivity is a separate stage.

In [ ]:
%pip -q install 'ultralytics>=8.3,<9' pyyaml
import torch, ultralytics
print('Ultralytics:', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU: enable a GPU runtime for training')

## Upload dataset ZIP
Required folders at the ZIP root: `images/train`, `images/val`, `images/test`, and matching `labels/train`, `labels/val`, `labels/test`. Each image needs a same-stem `.txt` label. Each object line has 11 numbers: class, normalized box (4), keypoint 1 (x,y,visibility), keypoint 2 (x,y,visibility). Keep source/license records outside the training archive. Use a fresh runtime if replacing a previously extracted dataset.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, stat
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload one dataset ZIP.'
name, archive_bytes = next(iter(uploaded.items()))
assert name.lower().endswith('.zip'), 'Expected a ZIP file.'
root = Path('/content/circuit_dataset')
assert not root.exists(), 'Use a fresh runtime to avoid mixing datasets.'
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    assert sum(i.file_size for i in archive.infolist()) < 4_000_000_000, 'Dataset exceeds this notebook’s 4 GB extraction limit.'
    for entry in archive.infolist():
        target = (root / entry.filename).resolve()
        assert target.is_relative_to(root.resolve()), 'Unsafe archive path.'
        assert not stat.S_ISLNK(entry.external_attr >> 16), 'Symlinks are not allowed.'
    archive.extractall(root)
print('Extracted:', root)

In [ ]:
import yaml, hashlib, math
from PIL import Image
seen_hashes = {}
class_counts = [0, 0, 0]
for split in ('train', 'val', 'test'):
    images = [p for p in (root / 'images' / split).glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')]
    assert images, f'No images in {split}.'
    assert len({p.stem for p in images}) == len(images), f'Duplicate filename stems in {split}.'
    for image in images:
        with Image.open(image) as im:
            im.verify()
        digest = hashlib.sha256(image.read_bytes()).hexdigest()
        assert digest not in seen_hashes or seen_hashes[digest] == split, 'Identical image leaked across splits.'
        seen_hashes[digest] = split
        label = root / 'labels' / split / (image.stem + '.txt')
        assert label.exists(), f'Missing label: {label}'
        for line in label.read_text().splitlines():
            if not line.strip():
                continue
            v = list(map(float, line.split()))
            assert len(v) == 11 and all(math.isfinite(x) for x in v), f'Invalid pose label: {label}'
            assert v[0] in (0, 1, 2), f'Invalid class: {label}'
            assert all(0 <= v[i] <= 1 for i in (1,2,3,4,5,6,8,9)), f'Coordinates must be normalized: {label}'
            assert v[3] > 0 and v[4] > 0, f'Zero-size box: {label}'
            assert v[7] in (0,1,2) and v[10] in (0,1,2), f'Invalid visibility: {label}'
            if split == 'train':
                class_counts[int(v[0])] += 1
    print(split, len(images), 'images')
assert all(class_counts), f'Training split is missing a class: {class_counts}'
config = {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0:'resistor', 1:'led', 2:'wire'}, 'kpt_shape': [2,3]}
yaml_path = root / 'data.yaml'
yaml_path.write_text(yaml.safe_dump(config))
print('Labels validated. Still review near-duplicates, polarity, source grouping, and annotations manually.')

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n-pose.pt')
training = model.train(data=str(yaml_path), epochs=60, imgsz=960, batch=4, device=0 if torch.cuda.is_available() else 'cpu', project='/content/circuit_runs', name='terminal_pose', seed=42, patience=15, fliplr=0.0, flipud=0.0, mosaic=0.0)
best_path = Path(model.trainer.best)
print('Best weights:', best_path)

## Evaluate once on the held-out test split
Box and pose mAP are useful diagnostics, but do not establish correct hole assignment or circuit diagnosis. Also measure terminal-to-hole accuracy and per-fault precision/recall on independently labeled builds. Never report hand-corrected results as automatic predictions. Do not tune repeatedly on the test set.

In [ ]:
trained = YOLO(str(best_path))
metrics = trained.val(data=str(yaml_path), split='test', imgsz=960, device=0 if torch.cuda.is_available() else 'cpu')
print(metrics.results_dict)
sample = next(p for p in (root/'images'/'test').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png','.webp'))
prediction = trained.predict(str(sample), conf=0.35)[0]
from IPython.display import display
display(Image.fromarray(prediction.plot()[..., ::-1]))

In [ ]:
import json
summary = {'ultralytics_version': ultralytics.__version__, 'model':'yolo11n-pose', 'classes':config['names'], 'keypoints':['terminal_1_or_anode','terminal_2_or_cathode'], 'metrics':metrics.results_dict, 'notes':'Pose results only; not electrical fault accuracy.'}
Path('/content/training_summary.json').write_text(json.dumps(summary, indent=2))
files.download(str(best_path))
files.download('/content/training_summary.json')

## Connect to the app
Place the downloaded weights at `circuit-debugger/models/best.pt`. Install `ultralytics` in the backend environment and restart the server. Upload a photo and click **Suggest components with AI**. Review the proposals and enter the actual hole numbers in the connection table before checking. Automatic board calibration and hole snapping are not part of version 0.1.